# GPL domain adaptation on SciFact and NFCorpus

This notebook adapts a dense retriever to two biomedical [BEIR](https://github.com/beir-cellar/beir) corpora with **GPL** (*Generative Pseudo Labeling*, Wang et al., NAACL 2022), using the official [UKPLab/gpl](https://github.com/UKPLab/gpl) library.

Training is **fully unsupervised**: a T5 model generates the queries and a cross-encoder writes the labels. The human relevance judgements (`qrels/test.tsv`) are used only for evaluation.

![GPL pipeline](/app/pairs/FSC_gpl/notebooks/img/gpl_pipeline.png)

| Dataset | Documents | Queries | What it stresses |
|---|:-:|:-:|---|
| SciFact | 5,183 | 300 | scientific claims with high word overlap |
| NFCorpus | 3,633 | 323 | lay medical queries against abstracts |

Both datasets must be in BEIR format (`corpus.jsonl`, `queries.jsonl`, `qrels/test.tsv`) under `/app/data/<name>`.

The notebook runs the same `gpl.train` call twice:

1. **Smoke test**: a tiny configuration that checks the whole pipeline runs end to end in a few minutes.
2. **Real run**: enough data and steps for the metrics to mean something.

## Download models

Pre-download every model used in `gpl.train` so the training run does not stop to fetch weights. The model IDs here must match the ones passed to `gpl.train` exactly, otherwise the model is downloaded a second time.

In [ ]:
from huggingface_hub import snapshot_download

models = [
    "doc2query/msmarco-t5-base-v1",                      # generator
    "sentence-transformers/msmarco-distilbert-base-v3",  # retriever 1
    "sentence-transformers/msmarco-MiniLM-L6-v3",        # retriever 2
    "cross-encoder/ms-marco-MiniLM-L6-v2",               # cross-encoder
    "BAAI/bge-small-en-v1.5",                           # student
]

for model_name in models:
    snapshot_download(repo_id=model_name, max_workers=8)
    print(f"Downloaded: {model_name}")

## `gpl.train` arguments

What each argument is, and what passing it (or its value) implies. The stage numbers refer to the diagram above.

### Paths

| Argument | What it is | What it implies |
|---|---|---|
| `path_to_generated_data` | Folder where GPL writes everything it generates: corpus copy, queries, negatives, pseudo-labels. | Works as a **cache**: each stage is skipped if its output file already exists there. If you change an argument and reuse the folder, GPL silently reuses the old data. Use a different folder per configuration, or delete the files from the stage you changed onwards. |
| `evaluation_data` | Path to the original BEIR dataset. | The source of the corpus that gets copied into `path_to_generated_data`, and the test set used in stage 5. |
| `output_dir` | Where the trained student is saved. | The final model ends up in `output_dir/<gpl_steps>/`. If that folder already exists, training is skipped. |
| `evaluation_output` | Where the metrics are written. | Creates `results.json` with nDCG, MAP, Recall, Precision and MRR at k = 1, 3, 5, 10, 20, 100. |

### Models

| Argument | What it is | What it implies |
|---|---|---|
| `generator` | Seq2seq model that writes queries for a passage (stage 1). | `doc2query/msmarco-t5-base-v1` is a T5-base (220M parameters) trained on MS MARCO. It is the largest model in the pipeline, and it generates token by token, so stage 1 is usually the slowest data stage. |
| `retrievers` | Bi-encoders that retrieve hard-negative candidates (stage 2). | Two different models give a more varied pool of negatives. |
| `retriever_score_functions` | Similarity used by each retriever, in the same order. | Must match how each retriever was trained. Both MS MARCO v3 models were trained with `"cos_sim"`. |
| `cross_encoder` | The "teacher" that scores (query, document) pairs (stage 3). | It reads query and document together, so it is more accurate than a bi-encoder but far too slow to use for search. GPL distils its judgement into the student. |
| `base_ckpt` | Starting checkpoint of the student (stage 4). | BAAI/bge-small-en-v1.5 (33M) is a sentence embedding model (bi-encoder) designed for semantic feature extraction, semantic similarity computation, and information retrieval. |
| `gpl_score_function` | Similarity the student uses to score query–document pairs. | `"dot"` matches the cross-encoder margins, which are unbounded. |

### Data size

| Argument | What it is | What it implies |
|---|---|---|
| `new_size` | Number of documents randomly sampled from the corpus to generate training data. | Passing a number turns on sampling; `None` uses the full corpus. Evaluation always uses the full corpus. Constraint: `new_size × queries_per_passage ≥ batch_size_gpl`. |
| `queries_per_passage` | Synthetic queries generated per document (QPP). | Total unique training queries = `new_size × QPP`. Linear cost in stage 1. Too many per document produces near-duplicates. |
| `negatives_per_query` | Top-k documents each retriever returns per query. | One negative is drawn at random from the union of both lists for every training tuple. Almost no compute cost; it controls **difficulty**. A small k gives very hard negatives, some of which are actually relevant (false negatives); the cross-encoder corrects them by giving them a margin near 0. A large k gives more variety. |

### Compute

| Argument | What it is | What it implies |
|---|---|---|
| `batch_size_generation` | Documents per batch in the T5 generator. | Each batch produces `batch_size_generation × QPP` sequences. Only affects speed and GPU memory in stage 1, not the generated data. |
| `batch_size_gpl` | (query, positive, negative) tuples per step. | Used in stages 3 and 4: 2·B cross-encoder pairs and 3·B student forward/backward passes per step. GPU memory per step grows with it. MarginMSE does not use in-batch negatives, so B does not change the loss, only throughput and gradient noise. |
| `gpl_steps` | Optimisation steps. | Total training data = `gpl_steps × batch_size_gpl` tuples. |

### Behaviour

| Argument | What it is | What it implies |
|---|---|---|
| `qgen_prefix` | Prefix for the synthetic query files. | Produces `query_gen-queries.jsonl` and `query_gen-qrels/`. Lets synthetic queries live next to real ones without overwriting them. |
| `do_evaluation` | Run stage 5 after training. | With `True`, `evaluation_data` must be a valid BEIR dataset with qrels for `eval_split`. With `False`, the run stops after training. |
| `eval_split` | Which qrels split to evaluate on. | `"test"` uses the SciFact and NFCorpus real test queries. |
| `use_train_qrels` | Use real training queries instead of generating them. | `False` keeps GPL unsupervised. `True` skips stage 1 and uses labelled data, so the run is no longer unsupervised. |

## 1. Smoke test

**Goal:** check that all five stages run without errors (paths, BEIR format, GPU memory, downloads) in a few minutes.

**The smoke test metrics are meaningless.** With 100 steps the learning rate only reaches 10 % of its peak (the warmup is 1,000 steps), so the student barely moves from its starting point.

| Argument | Value | What it implies |
|---|:-:|---|
| `new_size` | 100 | Only 100 documents (~2–3 % of each corpus), so generation and mining are almost instant. |
| `queries_per_passage` | 1 | 100 synthetic queries in total. |
| `negatives_per_query` | 5 | Top-5 per retriever (5 % of the 100-document sample). |
| `batch_size_generation` | 3 | 3 documents × 1 query = 3 sequences per T5 batch. |
| `batch_size_gpl` | 4 | Fits on almost any GPU. 100 × 1 ≥ 4 satisfies the library's assert. |
| `gpl_steps` | 100 | 100 × 4 = 400 pseudo-labelled training tuples. |

In this configuration the slowest stage is usually **evaluation**: it encodes the full corpus (5,183 / 3,633 documents) no matter how small the training was.

In [ ]:
import gpl

data_dict = {"scifact": "/app/data/scifact", "nfcorpus": "/app/data/nfcorpus"}

for dataset_key, dataset in data_dict.items():

    print(f"Starting GPL smoke test: {dataset_key}")

    gpl.train(
        path_to_generated_data=f"/app/pairs/FSC_gpl/models/gpl/smoke/{dataset_key}",

        # Student
        base_ckpt= "BAAI/bge-small-en-v1.5",  
        gpl_score_function="dot",

        # Smoke test
        gpl_steps=100,
        new_size=100,
        queries_per_passage=1,
        batch_size_generation=3,
        batch_size_gpl=4,
        negatives_per_query=5,

        # Output
        output_dir=f"/app/pairs/FSC_gpl/runs/models/smoke/{dataset_key}",
        evaluation_data=dataset,
        evaluation_output=f"/app/pairs/FSC_gpl/runs/evaluation/smoke/{dataset_key}",

        # Generator
        generator="doc2query/msmarco-t5-base-v1",

        # Hard negative mining
        retrievers=[
            "sentence-transformers/msmarco-distilbert-base-v3",
            "sentence-transformers/msmarco-MiniLM-L6-v3",
        ],
        retriever_score_functions=["cos_sim", "cos_sim"],

        # Pseudo labeling
        cross_encoder="cross-encoder/ms-marco-MiniLM-L6-v2",

        qgen_prefix="query_gen",

        # Evaluation
        do_evaluation=True,
        eval_split="test",
        use_train_qrels=False,
    )

    print(f"Smoke test finished: {dataset_key}")

## 2. Real run

Same call, with only the six data and compute arguments changed. It writes to `.../real/...` folders, so nothing from the smoke test is reused.

| Argument | Smoke | Real | Paper | Stages | What the change implies for compute and time |
|---|:-:|:-:|:-:|:-:|---|
| `new_size` | 100 | 1,000 | auto | 1, 2 | 10× more documents: 19 % of SciFact and 28 % of NFCorpus. Better coverage of domain vocabulary, and negatives come from a larger pool. |
| `queries_per_passage` | 1 | 3 | auto | 1, 2 | Combined with `new_size`: 3,000 queries instead of 100, so stage 1 does **30×** the work. |
| `negatives_per_query` | 5 | 50 | 50 | 2 | Negligible extra cost. Still 5 % of the sampled corpus, but a larger and more varied pool of negatives. |
| `batch_size_generation` | 3 | 2 | — | 1 | Lowered because QPP tripled: 2 × 3 = 6 sequences per batch instead of 3, keeping T5 memory in check. Raise it if your GPU has room; it is the cheapest way to speed up stage 1. |
| `batch_size_gpl` | 4 | 8 | 32 | 3, 4 | Twice the GPU memory per step. Each step processes twice as many tuples, but on a GPU it takes less than twice as long. |
| `gpl_steps` | 100 | 1,000 | 140,000 | 3, 4 | Combined with `batch_size_gpl`: 8,000 tuples instead of 400, so stages 3 and 4 do **20×** the work. |

**Total work per dataset**

| Stage | Smoke | Real | Ratio |
|---|--:|--:|:-:|
| 1. Queries generated | 100 | 3,000 | ×30 |
| 2. Texts encoded per retriever | 200 | 4,000 | ×20 |
| 3. Cross-encoder pairs scored | 800 | 16,000 | ×20 |
| 4. Student sequences (fwd + bwd) | 1,200 | 24,000 | ×20 |
| 5. Evaluation (docs + queries) | 5,483 / 3,956 | 5,483 / 3,956 | ×1 |

In the real run the time is dominated by query generation and stages 3–4, whereas in the smoke test it is dominated by evaluation. For reference, the paper setup (~250K queries, 140,000 × 32 ≈ 4.5M tuples) does about **560×** the stage 3–4 work of this real run.


In [ ]:
for dataset_key, dataset in data_dict.items():

    print(f"Starting GPL real run: {dataset_key}")

    gpl.train(
        path_to_generated_data=f"/app/pairs/FSC_gpl/runs/generated_gpl/real/{dataset_key}",

        # Student
        base_ckpt= "BAAI/bge-small-en-v1.5",
        gpl_score_function="dot",

        # Real run
        gpl_steps=1000,
        new_size=1000,  # None = full corpus
        queries_per_passage=3,
        batch_size_generation=2,
        batch_size_gpl=8,
        negatives_per_query=50,

        # Output
        output_dir=f"/app/pairs/FSC_gpl/runs/models/real/{dataset_key}",
        evaluation_data=dataset,
        evaluation_output=f"/app/pairs/FSC_gpl/runs/evaluation/real/{dataset_key}",

        # Generator
        generator="doc2query/msmarco-t5-base-v1",

        # Hard negative mining
        retrievers=[
            "sentence-transformers/msmarco-distilbert-base-v3",
            "sentence-transformers/msmarco-MiniLM-L6-v3",
        ],
        retriever_score_functions=["cos_sim", "cos_sim"],

        # Pseudo labeling
        cross_encoder="cross-encoder/ms-marco-MiniLM-L6-v2",

        qgen_prefix="query_gen",

        # Evaluation
        do_evaluation=True,
        eval_split="test",
        use_train_qrels=False,
    )

    print(f"Real run finished: {dataset_key}")